In [0]:
%pip install -U mlflow transformers torch huggingface_hub sentencepiece accelerate 

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("model_name", embedding_model_list[0], embedding_model_list)
model_name = dbutils.widgets.get("model_name")

dbutils.widgets.dropdown("catalog", catalog_list[0], catalog_list)
catalog = dbutils.widgets.get("catalog")


In [0]:
embedding_model_dct[model_name]["model"]

In [0]:
import mlflow
import numpy as np
from sentence_transformers import SentenceTransformer
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types import ColSpec, Schema, TensorSpec

mlflow.set_registry_uri("databricks-uc")

# Prepare input example as a list of strings (not a Series)
input_example = ["مرحبا كيفك", "شو أخبارك"]
# Download the model
model = SentenceTransformer(
    embedding_model_dct[model_name]["model"]
)
# Optional: Infer signature from the model
# This helps MLflow understand the expected input/output format
try:
    example_output = model.encode(input_example)
    signature = infer_signature(input_example, example_output)
except Exception as e:
    print(f"Could not infer signature: {e}")
    signature = None

with mlflow.start_run():
    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path="model",
        registered_model_name=model_name,
        signature=signature,
        input_example=input_example  # Use the list directly
    )

In [0]:
model.save(f"/Volumes/{catalog}/bertopic/models/{model_name}")


In [0]:
import pandas as pd

# Load from Unity Catalog registry using version number
loaded = mlflow.pyfunc.load_model(f"models:/{model_name}/1")


In [0]:
loaded.encode(input_example)


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

w.serving_endpoints.create_and_wait(
    name="marbert-matryoshka-embeddings",
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=f"workspace.default.{model_name}",
                entity_version="1",
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
)

In [0]:
%sql
SELECT ai_query(
    'marbert-matryoshka-embeddings',
    'كيف الحال يا شباب'
) AS embedding;

In [0]:
  import os
  import requests
  DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

  workspace_url = "https://dbc-de54b796-a6c4.cloud.databricks.com/"
  endpoint_name = "marbertv2-incitement-detector"
  token = DATABRICKS_TOKEN

  payload = {
      "dataframe_records": [
          {"text": "انت يا عميل السفارات يا ابن الكلب حسابك عسير"}
      ]
  }

  resp = requests.post(
      f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations",
      headers={
          "Authorization": f"Bearer {token}",
          "Content-Type": "application/json",
      },
      json=payload,
      timeout=120,
  )

  print(resp.status_code)
  print(resp.json())

In [0]:
loaded.predict(pd.DataFrame({"text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]}))